In [1]:
# API documentation: https://wetterdienst.readthedocs.io/en/latest/usage/quickstart.html
from wetterdienst.provider.dwd.observation import DwdObservationRequest
import datetime as dt
from wetterdienst.provider.dwd.observation import DwdObservationMetadata
import pandas as pd
import polars as pl

# Look at all the data provided by dwd
metadata = DwdObservationMetadata.model_dump_json(indent=2)
#print(metadata)
# Find stations close to the Kirchentellinsfurt Wehr with relevant info
kirchentellinsfurt_wehr = (48.533, 9.183)

request_daily = DwdObservationRequest(
    parameters=[("daily", "climate_summary", "temperature_air_mean_2m"), ("daily", "climate_summary", "precipitation_amount"),
                ("daily", "climate_summary", "pressure_vapor_mean"), ("daily", "climate_summary", "humidity_relative_mean")],
    start=dt.datetime(2000, 1, 1),
    end=dt.datetime(2026, 8, 1)
)
# Get the stations within 50km of the Kirchentellinsfurt Wehr
stations = request_daily.filter_by_distance(latlon=kirchentellinsfurt_wehr, distance=50, unit="km")
stations_present_records = stations.df.filter(stations.df["end_timestamp"] > dt.datetime(2026, 8, 1, tzinfo=dt.timezone.utc))
pd.DataFrame(stations_present_records).to_csv("../data/intermediate/stations_climate_summary_50km_2000_2026.csv", index=False)
# Get the stations within 30km of the Kirchentellinsfurt Wehr
stations = request_daily.filter_by_distance(latlon=kirchentellinsfurt_wehr, distance=30, unit="km")
stations_present_records_30 = stations.df.filter(stations.df["end_timestamp"] > dt.datetime(2026, 8, 1, tzinfo=dt.timezone.utc))
pd.DataFrame(stations_present_records_30).to_csv("../data/intermediate/stations_climate_summary_30km_2000_2026.csv", index=False)


In [2]:

# Get the data download for the stations that are present in the 30km radius.
relevant_requests = request_daily.filter_by_station_id(station_id=["03278", "04931", "02074", "06275", "03402"])
data = relevant_requests.values().all().df.drop_nulls("value")
# Convert data to long format (parameters as columns, timestamps and station_id as rows) for easier analysis
data_long = data.pivot(
    index=["station_id", "timestamp"],
    on="parameter",
    values="value"
)
pd.DataFrame(data_long).to_csv("../data/intermediate/weather_data_daily_2000_2026.csv", index=False, header=data_long.columns)

In [9]:
# Count the number of observations per station_id and parameter
observation_counts = data.group_by(["station_id", "parameter"]).agg([pl.count("value").alias("observation_count")])
observation_counts # Note: no station has data for every day )


station_id,parameter,observation_count
str,str,u32
"""06275""","""temperature_air_mean_2m""",8067
"""03278""","""pressure_vapor_mean""",8305
"""02074""","""humidity_relative_mean""",9670
"""04931""","""precipitation_amount""",9706
"""02074""","""pressure_vapor_mean""",9670
…,…,…
"""06275""","""precipitation_amount""",8064
"""04931""","""humidity_relative_mean""",9623
"""02074""","""temperature_air_mean_2m""",9683


In [5]:

# Collect hourly data
request_hourly = DwdObservationRequest(
    parameters=[("hourly", "moisture", "temperature_air_2m"), ("hourly", "moisture", "pressure_vapor"), ("hourly", "moisture", "humidity_relative"), ("hourly", "precipitation", "precipitation_amount")],
    start=dt.datetime(2000, 1, 1),
    end=dt.datetime(2026, 8, 1)
)

relevant_requests_hourly = request_hourly.filter_by_station_id(station_id=["03278", "04931", "02074", "06275", "03402"])

data_hourly = relevant_requests_hourly.values().all().df.drop_nulls("value")
data_hourly_long = data_hourly.pivot(
    index=["station_id", "timestamp"],
    on="parameter",
    values="value"
)
pd.DataFrame(data_hourly_long).to_csv("../data/intermediate/weather_data_hourly_2000_2026.csv", index=False, header=data_hourly_long.columns)

In [6]:
# Count the number of observations per station_id and parameter
observation_counts_hourly = data_hourly.group_by(["station_id", "parameter"]).agg([pl.count("value").alias("observation_count")])
observation_counts_hourly # data not recorded every hour in time window


station_id,parameter,observation_count
str,str,u32
"""04931""","""temperature_air_2m""",230677
"""02074""","""precipitation_amount""",193454
"""03278""","""temperature_air_2m""",198894
"""03402""","""humidity_relative""",206228
"""03402""","""pressure_vapor""",206228
…,…,…
"""02074""","""temperature_air_2m""",193019
"""06275""","""pressure_vapor""",193096
"""03278""","""precipitation_amount""",199208


In [7]:
# Missingness is measured against a regular expected time grid because the API export contains observed rows only.
import matplotlib.pyplot as plt
from pathlib import Path
OUT_DIR = Path("../out")
OUT_DIR.mkdir(exist_ok=True)

def missingness_analysis(path, frequency, variables):
    data = pd.read_csv(path, parse_dates=["timestamp"])
    data["station_id"] = data["station_id"].astype(str).str.zfill(5)
    long = data.melt(id_vars=["station_id", "timestamp"], value_vars=variables, var_name="variable", value_name="value")
    summaries, monthly = [], []
    for (station, variable), group in long.groupby(["station_id", "variable"]):
        observed = pd.DatetimeIndex(group.loc[group["value"].notna(), "timestamp"].unique()).sort_values()
        if not len(observed):
            continue
        expected = pd.date_range(observed.min(), observed.max(), freq=frequency)
        present = pd.Series(expected.isin(observed), index=expected)
        missing = ~present
        runs = missing.groupby(missing.ne(missing.shift()).cumsum()).sum()
        gaps = runs[runs > 0]
        summaries.append({"station_id": station, "variable": variable, "first_observation": observed.min(), "last_observation": observed.max(), "expected_observations": len(expected), "observed_observations": int(present.sum()), "missing_observations": int(missing.sum()), "missing_percent": 100 * missing.mean(), "missing_periods": len(gaps), "longest_missing_periods": int(gaps.max()) if len(gaps) else 0, "frequency": frequency})
        frame = pd.DataFrame({"timestamp": expected, "observed": present.to_numpy()})
        frame["month"] = frame["timestamp"].dt.to_period("M").dt.to_timestamp()
        frame = frame.groupby("month", as_index=False)["observed"].mean()
        frame["station_id"], frame["variable"] = station, variable
        monthly.append(frame)
    return pd.DataFrame(summaries), pd.concat(monthly, ignore_index=True)

daily_summary, daily_monthly = missingness_analysis("../data/intermediate/weather_data_daily_2000_2026.csv", "D", ["temperature_air_mean_2m", "precipitation_amount", "humidity_relative_mean", "pressure_vapor_mean"])
hourly_summary, hourly_monthly = missingness_analysis("../data/intermediate/weather_data_hourly_2000_2026.csv", "h", ["temperature_air_2m", "precipitation_amount", "humidity_relative", "pressure_vapor"])
missing_summary = pd.concat([daily_summary, hourly_summary], ignore_index=True)
missing_summary.to_csv(OUT_DIR / "weather_missingness_summary.csv", index=False)
daily_summary.to_csv(OUT_DIR / "weather_daily_missingness_summary.csv", index=False)
hourly_summary.to_csv(OUT_DIR / "weather_hourly_missingness_summary.csv", index=False)

def missingness_heatmap(summary, title, filename):
    table = summary.pivot(index="variable", columns="station_id", values="missing_percent")
    fig, axis = plt.subplots(figsize=(10, 4.5))
    image = axis.imshow(table, cmap="YlOrRd", vmin=0, vmax=max(1, table.to_numpy().max()), aspect="auto")
    axis.set(title=title, xlabel="Station", ylabel="Variable", xticks=range(len(table.columns)), xticklabels=table.columns, yticks=range(len(table.index)), yticklabels=table.index)
    for row in range(table.shape[0]):
        for col in range(table.shape[1]):
            axis.text(col, row, f"{table.iloc[row, col]:.1f}%", ha="center", va="center", fontsize=8)
    fig.colorbar(image, ax=axis, label="Missing observations (%)")
    fig.tight_layout(); fig.savefig(OUT_DIR / filename, dpi=180, bbox_inches="tight"); plt.close(fig)

def completeness_heatmap(monthly, title, filename):
    table = (monthly.assign(observed=100 * monthly["observed"]).groupby(["station_id", "month"], as_index=False)["observed"].mean().pivot(index="station_id", columns="month", values="observed"))
    fig, axis = plt.subplots(figsize=(15, 3.5))
    image = axis.imshow(table, cmap="RdYlGn", vmin=0, vmax=100, aspect="auto", interpolation="nearest")
    step = max(1, len(table.columns) // 12); ticks = range(0, len(table.columns), step)
    axis.set(title=title, xlabel="Month", ylabel="Station", xticks=ticks, xticklabels=[table.columns[i].strftime("%Y-%m") for i in ticks], yticks=range(len(table.index)), yticklabels=table.index)
    axis.tick_params(axis="x", rotation=45); fig.colorbar(image, ax=axis, label="Observed data (%)")
    fig.tight_layout(); fig.savefig(OUT_DIR / filename, dpi=180, bbox_inches="tight"); plt.close(fig)

missingness_heatmap(hourly_summary, "Hourly weather missingness", "weather_hourly_missingness_heatmap.png")
missingness_heatmap(daily_summary, "Daily weather missingness", "weather_daily_missingness_heatmap.png")
completeness_heatmap(hourly_monthly, "Hourly weather completeness by month", "weather_hourly_completeness_by_month.png")
completeness_heatmap(daily_monthly, "Daily weather completeness by month", "weather_daily_completeness_by_month.png")
display(missing_summary.sort_values(["frequency", "missing_percent"], ascending=[True, False]))


,station_id,variable,first_observation,last_observation,expected_observations,observed_observations,missing_observations,missing_percent,missing_periods,longest_missing_periods,frequency
4,03278,humidity_relative_mean,2003-01-01,2026-08-01,8614,8305,309,3.587184,8,294,D
6,03278,pressure_vapor_mean,2003-01-01,2026-08-01,8614,8305,309,3.587184,8,294,D
5,03278,precipitation_amount,2003-01-01,2026-08-01,8614,8314,300,3.482703,4,294,D
7,03278,temperature_air_mean_2m,2003-01-01,2026-08-01,8614,8320,294,3.413049,1,294,D
12,04931,humidity_relative_mean,2000-01-01,2026-08-01,9710,9623,87,0.895984,3,80,D
14,04931,pressure_vapor_mean,2000-01-01,2026-08-01,9710,9623,87,0.895984,3,80,D
15,04931,temperature_air_mean_2m,2000-01-01,2026-08-01,9710,9632,78,0.803296,1,78,D
0,02074,humidity_relative_mean,2000-01-01,2026-08-01,9710,9670,40,0.411946,6,28,D
2,02074,pressure_vapor_mean,2000-01-01,2026-08-01,9710,9670,40,0.411946,6,28,D
3,02074,temperature_air_mean_2m,2000-01-01,2026-08-01,9710,9683,27,0.278064,1,27,D
